# 03 · Gold · `dec_fec_alimentador_mensal`
### V-Energy Grid 360 — Arquitetura Medalhão

Responde à Beatriz: *"a ANEEL cobra DEC e FEC da gente [...] quero acompanhar esses indicadores por alimentador, saber quais regiões concentram as piores interrupções e onde a manutenção preventiva dá mais retorno"* (requisito 5.1: calcular DEC e FEC por alimentador). Grava `v_energy.gold.dec_fec_alimentador_mensal`, seguindo o **contrato da Gold (tabela A8, versão v4)**, com as 14 colunas do contrato.

**Grão:** 1 linha = 1 alimentador × 1 mês, com **grade completa**: 122 alimentadores × 48 meses (`2022-07` a `2026-06`) = **5.856 linhas**. Mês sem interrupção aparece com DEC e FEC **zero**.

**Fontes (Gold):** `fato_interrupcao` (ocorrências, duração e causa) e `dim_uc` (UCs de cada alimentador e data de ligação).

**Fórmulas** (simplificadas, conforme o glossário do case: cada ocorrência = 1 UC interrompida):

| Coluna | Regra |
|---|---|
| `qtd_ucs_alimentador_mes` | UCs do alimentador com `data_ligacao` até o último dia do mês (denominador) |
| `horas_interrompidas` | soma de `duracao_min` ÷ 60 (= UC × horas sem luz) |
| `dec_horas` | `horas_interrompidas ÷ qtd_ucs_alimentador_mes` |
| `fec` | `qtd_interrupcoes ÷ qtd_ucs_alimentador_mes` |
| `dec_12m_horas`, `fec_12m` | soma do DEC e do FEC mensais dos 12 meses até este mês (o mês e os 11 anteriores) |
| `causa_principal` | causa com mais horas no mês |
| `ranking_dec_mes` | posição do alimentador no mês por `dec_horas` (1 = pior) |

**Decisões deste notebook:**
- **`dec_12m_horas` e `fec_12m` são nulos nos 11 primeiros meses** (`2022-07` a `2023-05`): a base começa em `2022-07`, e um acumulado de menos de 12 meses ficaria menor que o real e enganaria o ranking. O primeiro valor é `2023-06`.
- **`causa_principal` considera só as causas informadas.** As 5.400 ocorrências sem causa (log O1) entram nas horas e no DEC, mas não disputam a causa principal: "sem causa" não ajuda a decidir onde fazer manutenção. É nula em mês sem interrupção (e em mês só com ocorrências sem causa, se houver). Empate em horas: vence a de nome menor, para o resultado ser estável.
- **`ranking_dec_mes`:** desempate pelas horas interrompidas (maior primeiro) e pelo código do alimentador. Cada mês tem as posições de 1 a 122, sem repetição.
- **Horas evitáveis são um piso:** vêm de `causa_evitavel` da `fato_interrupcao`, em que a causa nula é `false`.
- **`OSCILACAO` entra** no DEC e no FEC (suposição a registrar no README; ver a `fato_interrupcao`).

**Atenção ao denominador pequeno (contrato):** alimentadores com poucas UCs têm DEC alto com poucas ocorrências. A tela deve mostrar o DEC **junto** com `qtd_ucs_alimentador_mes` e `horas_interrompidas`. A seção 9 mostra o efeito.

**Pré-requisito:** `gold.fato_interrupcao` e `gold.dim_uc` gravadas.

## 1. Parâmetros do ambiente

In [0]:
catalog = "v_energy"
bronze_schema = f"{catalog}.bronze"
silver_schema = f"{catalog}.silver"
gold_schema = f"{catalog}.gold"
landing_path = f"/Volumes/{catalog}/bronze/landing"

from pyspark.sql import functions as F, Row, Window
from datetime import datetime

tabela_intr = f"{gold_schema}.fato_interrupcao"
tabela_dim = f"{gold_schema}.dim_uc"
tabela_gold = f"{gold_schema}.dec_fec_alimentador_mensal"
tabela_dq_log = f"{silver_schema}.dq_log"           # mesmo log das Silvers e das outras Golds

MES_INI, MES_FIM = "2022-07", "2026-06"
PRIMEIRO_MES_12M = "2023-06"                         # primeiro mês com 12 meses de histórico

def n_br(n):
    return f"{n:,}".replace(",", ".")

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {gold_schema}")
print(f"origem:  {tabela_intr}, {tabela_dim}")
print(f"destino: {tabela_gold}")

## 2. Funções de checagem de qualidade

Mesmas funções das Silvers e das outras Golds, com as mesmas colunas no log. Os resultados vão para `silver.dq_log` (sempre), e a tabela só é gravada se **todas** as checagens passarem.

In [0]:
dq_results = []

def dq_check(table_name: str, check_name: str, df, condition):
    """Executa uma checagem de qualidade: conta quantas linhas violam a condição esperada."""
    total = df.count()
    failed = df.filter(~condition).count()
    passed = failed == 0
    dq_results.append(Row(table_name=table_name, check_name=check_name, total_rows=total,
                          failed_rows=failed, passed=passed, checked_at=datetime.now()))
    print(f"[{'PASS' if passed else 'FAIL'}] {table_name} | {check_name} | {failed}/{total} linhas falharam")

def dq_check_unique(table_name: str, check_name: str, df, key_cols: list):
    """Checagem de qualidade específica para unicidade de chave."""
    total = df.count()
    dupes = df.groupBy(*key_cols).count().filter("count > 1").count()
    passed = dupes == 0
    dq_results.append(Row(table_name=table_name, check_name=check_name, total_rows=total,
                          failed_rows=dupes, passed=passed, checked_at=datetime.now()))
    print(f"[{'PASS' if passed else 'FAIL'}] {table_name} | {check_name} | {dupes} chaves duplicadas de {total} linhas")

def dq_check_contagem(table_name: str, check_name: str, df, esperado: int):
    """Checagem de contagem total: passa só se o número de linhas for exatamente o esperado."""
    total = df.count()
    diferenca = abs(total - esperado)
    passed = diferenca == 0
    dq_results.append(Row(table_name=table_name, check_name=check_name, total_rows=total,
                          failed_rows=diferenca, passed=passed, checked_at=datetime.now()))
    print(f"[{'PASS' if passed else 'FAIL'}] {table_name} | {check_name} | {n_br(total)} linhas (esperado {n_br(esperado)})")

def dq_check_valor(table_name: str, check_name: str, total_rows: int, obtido, esperado):
    """Compara um valor calculado com o esperado (soma que precisa bater com a fato ou com o case)."""
    passed = obtido == esperado
    dq_results.append(Row(table_name=table_name, check_name=check_name, total_rows=total_rows,
                          failed_rows=0 if passed else 1, passed=passed, checked_at=datetime.now()))
    print(f"[{'PASS' if passed else 'FAIL'}] {table_name} | {check_name} | obtido {obtido} (esperado {esperado})")

## 3. Leitura das fontes e conferência das colunas

Contagens esperadas: **180.000** interrupções e **45.000** UCs, em **122** alimentadores.

**Um município por alimentador:** o contrato diz que cada alimentador atende um único município (o código `ALM-XXX-000` traz o município). O notebook confere na `dim_uc` antes de usar.

In [0]:
intr = spark.table(tabela_intr)
dim = spark.table(tabela_dim)

exigidas = {
    tabela_intr: ["id_ocorrencia", "id_uc", "alimentador", "causa", "causa_evitavel", "mes_referencia", "duracao_min"],
    tabela_dim: ["id_uc", "alimentador", "municipio", "data_ligacao"],
}
for tabela, colunas in exigidas.items():
    faltando = [c for c in colunas if c not in spark.table(tabela).columns]
    if faltando:
        raise Exception(f"{tabela} não tem as colunas {faltando} (DESCRIBE {tabela}).")

n_intr, n_dim = intr.count(), dim.count()
print(f"interrupções: {n_br(n_intr)} | UCs: {n_br(n_dim)}")   # esperado: 180.000 | 45.000

alim_mun = dim.groupBy("alimentador").agg(F.countDistinct("municipio").alias("n_mun"), F.first("municipio").alias("municipio"))
print("alimentadores:", alim_mun.count())                                              # esperado: 122
print("alimentadores com mais de um município:", alim_mun.filter("n_mun > 1").count())  # esperado: 0

## 4. A grade: 122 alimentadores × 48 meses, e o denominador

**Grade completa:** todo par alimentador × mês existe, mesmo sem interrupção. Sem a grade, o mês sem ocorrência simplesmente sumiria da tabela, e a média e o acumulado de 12 meses ficariam errados.

**Denominador (`qtd_ucs_alimentador_mes`):** UCs do alimentador com `data_ligacao` até o último dia do mês. A base cresce (44.741 UCs faturadas no primeiro ano, 45.000 no fim), então o denominador também cresce.

**Conferência:** 5.856 linhas; nenhum alimentador com zero UCs; 45.000 UCs somadas em `2026-06`.

In [0]:
meses = (spark.sql(f"SELECT explode(sequence(to_date('{MES_INI}-01'), to_date('{MES_FIM}-01'), interval 1 month)) AS ini")
         .select(F.date_format("ini", "yyyy-MM").alias("mes_referencia"), F.last_day("ini").alias("fim_mes")))
grade = alim_mun.select("alimentador", "municipio").crossJoin(meses)
print("meses:", meses.count(), "| linhas da grade:", n_br(grade.count()))   # esperado: 48 | 5.856

ucs_mes = (grade.join(dim.select("alimentador", "data_ligacao"), "alimentador", "left")
           .groupBy("alimentador", "mes_referencia")
           .agg(F.sum(F.when(F.col("data_ligacao") <= F.col("fim_mes"), 1).otherwise(0)).alias("qtd_ucs_alimentador_mes")))
print("pares com zero UCs:", ucs_mes.filter("qtd_ucs_alimentador_mes = 0").count())   # esperado: 0
print("UCs em 2026-06:", n_br(ucs_mes.filter(f"mes_referencia = '{MES_FIM}'").agg(F.sum("qtd_ucs_alimentador_mes")).first()[0]))   # esperado: 45.000

## 5. Interrupções, horas e causa principal por alimentador e mês

- **Contagens e horas:** ocorrências, minutos e as mesmas medidas só das causas evitáveis.
- **Causa principal:** horas por causa **informada**; vence a de mais horas (empate: nome menor).

**Conferência:** soma das ocorrências = 180.000 e soma dos minutos = a da `fato_interrupcao`.

In [0]:
intr_mes = (intr.groupBy("alimentador", "mes_referencia").agg(
    F.count("*").alias("qtd_interrupcoes"),
    F.sum("duracao_min").alias("minutos"),
    F.sum(F.when(F.col("causa_evitavel"), 1).otherwise(0)).alias("qtd_interrupcoes_evitaveis"),
    F.sum(F.when(F.col("causa_evitavel"), F.col("duracao_min")).otherwise(0)).alias("minutos_evitaveis"),
))

w_causa = Window.partitionBy("alimentador", "mes_referencia").orderBy(F.desc("min_causa"), F.asc("causa"))
causa_mes = (intr.filter(F.col("causa").isNotNull())
             .groupBy("alimentador", "mes_referencia", "causa").agg(F.sum("duracao_min").alias("min_causa"))
             .withColumn("rn", F.row_number().over(w_causa)).filter("rn = 1")
             .select("alimentador", "mes_referencia", F.col("causa").alias("causa_principal")))

tot_intr = intr_mes.agg(F.sum("qtd_interrupcoes").alias("n"), F.sum("minutos").alias("m")).first()
print("ocorrências:", n_br(tot_intr["n"]), "| horas:", round(tot_intr["m"] / 60, 1))   # esperado: 180.000 | 356.534,8

# Informativo: ocorrências em UCs ainda não ligadas no mês (entrariam no numerador sem estar no denominador)
antes = (intr.join(dim.select("id_uc", "data_ligacao"), "id_uc")
         .filter(F.last_day(F.to_date(F.concat("mes_referencia", F.lit("-01")))) < F.col("data_ligacao")).count())
print("ocorrências antes da ligação da UC:", n_br(antes))   # esperado: 0 ou muito poucas

## 6. DEC, FEC, acumulado de 12 meses e ranking

- **`dec_horas` e `fec`:** horas e ocorrências do mês ÷ UCs do alimentador no mês. Com a grade, mês sem interrupção dá zero.
- **Acumulado de 12 meses:** soma do DEC e do FEC mensais do mês e dos 11 anteriores (janela por alimentador, ordenada por mês). É a forma como a ANEEL apura o indicador anual: a soma dos valores mensais. Nulo antes de `2023-06`.
- **Ranking do mês:** por `dec_horas` decrescente, depois `horas_interrompidas` decrescente e o código do alimentador.

Os cálculos usam `double` e o resultado é gravado em `decimal` com 4 casas no DEC e no FEC.

In [0]:
base = (grade.select("alimentador", "municipio", "mes_referencia")
        .join(ucs_mes, ["alimentador", "mes_referencia"], "left")
        .join(intr_mes, ["alimentador", "mes_referencia"], "left")
        .join(causa_mes, ["alimentador", "mes_referencia"], "left")
        .fillna(0, subset=["qtd_interrupcoes", "minutos", "qtd_interrupcoes_evitaveis", "minutos_evitaveis"])
        .withColumn("horas_interrompidas", F.col("minutos") / 60.0)
        .withColumn("horas_causas_evitaveis", F.col("minutos_evitaveis") / 60.0)
        .withColumn("dec_horas", F.expr("try_divide(horas_interrompidas, qtd_ucs_alimentador_mes)"))
        .withColumn("fec", F.expr("try_divide(qtd_interrupcoes, qtd_ucs_alimentador_mes)")))

w12 = Window.partitionBy("alimentador").orderBy("mes_referencia").rowsBetween(-11, 0)
w_rank = Window.partitionBy("mes_referencia").orderBy(F.desc("dec_horas"), F.desc("horas_interrompidas"), F.asc("alimentador"))
tem_12m = F.col("mes_referencia") >= PRIMEIRO_MES_12M

dec_fec = (base
           .withColumn("dec_12m_horas", F.when(tem_12m, F.sum("dec_horas").over(w12)))
           .withColumn("fec_12m", F.when(tem_12m, F.sum("fec").over(w12)))
           .withColumn("ranking_dec_mes", F.row_number().over(w_rank)))

df_gold = dec_fec.select(
    "alimentador", "municipio", "mes_referencia",
    F.col("qtd_ucs_alimentador_mes").cast("int").alias("qtd_ucs_alimentador_mes"),
    F.col("qtd_interrupcoes").cast("int").alias("qtd_interrupcoes"),
    F.round("horas_interrompidas", 2).cast("decimal(12,2)").alias("horas_interrompidas"),
    F.col("qtd_interrupcoes_evitaveis").cast("int").alias("qtd_interrupcoes_evitaveis"),
    F.round("horas_causas_evitaveis", 2).cast("decimal(12,2)").alias("horas_causas_evitaveis"),
    F.round("dec_horas", 4).cast("decimal(10,4)").alias("dec_horas"),
    F.round("fec", 4).cast("decimal(10,4)").alias("fec"),
    F.round("dec_12m_horas", 4).cast("decimal(10,4)").alias("dec_12m_horas"),
    F.round("fec_12m", 4).cast("decimal(10,4)").alias("fec_12m"),
    "causa_principal",
    F.col("ranking_dec_mes").cast("int").alias("ranking_dec_mes"),
)
print("colunas:", len(df_gold.columns))   # esperado: 14
df_gold.printSchema()
display(df_gold.filter("mes_referencia = '2026-06'").orderBy("ranking_dec_mes").limit(10))

## 7. Checagens de qualidade

| Checagem | Protege |
|---|---|
| 5.856 linhas; `(alimentador, mes_referencia)` único; 122 alimentadores e 48 meses | a grade completa |
| Colunas não nulas (as de 12 meses e a causa principal têm regras próprias) | junções |
| `qtd_ucs_alimentador_mes` > 0 e nunca diminui no alimentador; 45.000 UCs em `2026-06` | o denominador |
| **Soma das ocorrências = 180.000; soma das horas e das horas evitáveis = as da `fato_interrupcao`** | nada se perdeu nem duplicou na grade |
| `dec_horas` = horas ÷ UCs e `fec` = ocorrências ÷ UCs; zero quando não há interrupção | as fórmulas |
| Evitáveis <= total (ocorrências e horas) | coerência |
| `dec_12m_horas` e `fec_12m` nulos antes de `2023-06` e preenchidos depois; >= DEC e FEC do mês | o acumulado |
| `causa_principal` nula quando não há interrupção, e nas 6 causas quando há | a causa |
| `ranking_dec_mes` de 1 a 122 em cada mês, sem repetição; ranking 1 com o maior DEC do mês | o ranking |
| **17.844 ocorrências em `2025-07`** e os três alimentadores do evento no topo | o evento climático conhecido |

In [0]:
t = "gold.dec_fec_alimentador_mensal"
dq_results.clear()

CAUSAS = ["CLIMATICA", "SOBRECARGA", "NAO IDENTIFICADA", "VEGETACAO", "TERCEIROS", "FALHA DE EQUIPAMENTO"]
def ok(cond):
    return F.coalesce(cond, F.lit(False))   # nulo também falha

g = df_gold
n_g = g.count()

# A grade
dq_check_contagem(t, "linhas = 122 alimentadores x 48 meses = 5.856", g, 5_856)
dq_check_unique(t, "(alimentador, mes_referencia) único", g, ["alimentador", "mes_referencia"])
dq_check_valor(t, "alimentadores distintos = 122", n_g, g.select("alimentador").distinct().count(), 122)
dq_check_valor(t, "meses distintos = 48", n_g, g.select("mes_referencia").distinct().count(), 48)
dq_check(t, "alimentador no padrão ALM-XXX-000", g, ok(F.col("alimentador").rlike("^ALM-[A-Z]{3}-[0-9]{3}$")))
dq_check_valor(t, "um município por alimentador", n_g, alim_mun.filter("n_mun > 1").count(), 0)

for coluna in ["alimentador", "municipio", "mes_referencia", "qtd_ucs_alimentador_mes", "qtd_interrupcoes", "horas_interrompidas",
               "qtd_interrupcoes_evitaveis", "horas_causas_evitaveis", "dec_horas", "fec", "ranking_dec_mes"]:
    dq_check(t, f"{coluna} não nulo", g, F.col(coluna).isNotNull())

# O denominador
dq_check(t, "qtd_ucs_alimentador_mes > 0", g, ok(F.col("qtd_ucs_alimentador_mes") > 0))
w_alim = Window.partitionBy("alimentador").orderBy("mes_referencia")
dq_check(t, "qtd_ucs_alimentador_mes não diminui no alimentador",
         g.withColumn("ant", F.lag("qtd_ucs_alimentador_mes").over(w_alim)),
         F.col("ant").isNull() | (F.col("qtd_ucs_alimentador_mes") >= F.col("ant")))
dq_check_valor(t, "UCs somadas em 2026-06 = 45.000", n_g,
               g.filter(f"mes_referencia = '{MES_FIM}'").agg(F.sum("qtd_ucs_alimentador_mes")).first()[0], 45_000)

# Reconciliação com a fato_interrupcao (minutos, para não depender do arredondamento das horas)
fato_tot = intr.agg(F.count("*").alias("n"), F.sum("duracao_min").alias("m"),
                    F.sum(F.when(F.col("causa_evitavel"), 1).otherwise(0)).alias("ne"),
                    F.sum(F.when(F.col("causa_evitavel"), F.col("duracao_min")).otherwise(0)).alias("me")).first()
g_tot = dec_fec.agg(F.sum("qtd_interrupcoes").alias("n"), F.sum("minutos").alias("m"),
                    F.sum("qtd_interrupcoes_evitaveis").alias("ne"), F.sum("minutos_evitaveis").alias("me")).first()
dq_check_valor(t, "soma das ocorrências = fato_interrupcao (180.000)", n_g, g_tot["n"], fato_tot["n"])
dq_check_valor(t, "soma dos minutos = fato_interrupcao", n_g, g_tot["m"], fato_tot["m"])
dq_check_valor(t, "soma das ocorrências evitáveis = fato_interrupcao", n_g, g_tot["ne"], fato_tot["ne"])
dq_check_valor(t, "soma dos minutos evitáveis = fato_interrupcao", n_g, g_tot["me"], fato_tot["me"])

# As fórmulas (tolerância do arredondamento para 4 casas)
dq_check(t, "dec_horas = horas_interrompidas / UCs", g,
         ok(F.abs(F.col("dec_horas") - F.col("horas_interrompidas") / F.col("qtd_ucs_alimentador_mes")) <= 0.001))   # as horas já vêm arredondadas em 2 casas
dq_check(t, "fec = qtd_interrupcoes / UCs", g,
         ok(F.abs(F.col("fec") - F.col("qtd_interrupcoes") / F.col("qtd_ucs_alimentador_mes")) <= 0.0002))
dq_check(t, "sem interrupção: DEC, FEC e horas = 0", g,
         (F.col("qtd_interrupcoes") > 0) | ((F.col("dec_horas") == 0) & (F.col("fec") == 0) & (F.col("horas_interrompidas") == 0)))
dq_check(t, "evitáveis <= total (ocorrências e horas)", g,
         (F.col("qtd_interrupcoes_evitaveis") <= F.col("qtd_interrupcoes")) & (F.col("horas_causas_evitaveis") <= F.col("horas_interrompidas")))

# Acumulado de 12 meses
dq_check(t, f"12 meses: nulos antes de {PRIMEIRO_MES_12M} e preenchidos depois", g,
         F.when(F.col("mes_referencia") < PRIMEIRO_MES_12M, F.col("dec_12m_horas").isNull() & F.col("fec_12m").isNull())
          .otherwise(F.col("dec_12m_horas").isNotNull() & F.col("fec_12m").isNotNull()))
dq_check(t, "12 meses >= mês (DEC e FEC)", g.filter(F.col("dec_12m_horas").isNotNull()),
         (F.col("dec_12m_horas") >= F.col("dec_horas") - 0.0001) & (F.col("fec_12m") >= F.col("fec") - 0.0001))

# Causa principal
dq_check(t, "causa_principal nula quando não há interrupção", g.filter("qtd_interrupcoes = 0"), F.col("causa_principal").isNull())
dq_check(t, "causa_principal nas 6 causas quando preenchida", g,
         F.col("causa_principal").isNull() | ok(F.col("causa_principal").isin(*CAUSAS)))

# Ranking
dq_check_unique(t, "ranking_dec_mes único em cada mês", g, ["mes_referencia", "ranking_dec_mes"])
dq_check(t, "ranking_dec_mes entre 1 e 122", g, F.col("ranking_dec_mes").between(1, 122))
maior = g.groupBy("mes_referencia").agg(F.max("dec_horas").alias("max_dec"))
dq_check(t, "ranking 1 = maior DEC do mês", g.filter("ranking_dec_mes = 1").join(maior, "mes_referencia"),
         F.col("dec_horas") == F.col("max_dec"))

# O evento de julho de 2025
dq_check_valor(t, "ocorrências em 2025-07 = 17.844", n_g,
               g.filter("mes_referencia = '2025-07'").agg(F.sum("qtd_interrupcoes")).first()[0], 17_844)
top3 = {r["alimentador"] for r in g.filter("mes_referencia = '2025-07'").orderBy(F.desc("qtd_interrupcoes")).limit(3).collect()}
dq_check_valor(t, "2025-07: os 3 alimentadores com mais ocorrências são os do evento", n_g,
               sorted(top3), sorted(["ALM-REC-008", "ALM-SAO-004", "ALM-JAB-004"]))
dq_check_valor(t, "14 colunas, como no contrato", n_g, len(df_gold.columns), 14)

## 8. Registro das checagens e gravação, só se todas passarem

O log vai **sempre** para `silver.dq_log` (`table_name = gold.dec_fec_alimentador_mensal`); a tabela só é gravada se **todas** as checagens passarem.

In [0]:
spark.createDataFrame(dq_results).write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(tabela_dq_log)

falhas = [r for r in dq_results if not r.passed]
if falhas:
    print(f"{len(falhas)} checagem(ns) falharam. {tabela_gold} NÃO foi gravada:")
    for r in falhas:
        print(f"  - {r.check_name}: {n_br(r.failed_rows)} falha(s) em {n_br(r.total_rows)} linhas")
    raise Exception(f"{tabela_gold} não gravada: {len(falhas)} checagem(ns) de qualidade falharam (veja silver.dq_log)")

df_gold.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(tabela_gold)
print(f"Todas as {len(dq_results)} checagens passaram. {tabela_gold} gravada: {n_br(spark.table(tabela_gold).count())} linhas")

## 9. Consultas de conferência: o que a Beatriz vê

1. **Os piores alimentadores dos últimos 12 meses** (`dec_12m_horas` em `2026-06`), com UCs e horas, para mostrar o efeito do denominador pequeno.
2. **Onde a manutenção preventiva dá mais retorno:** horas de causas evitáveis nos últimos 12 meses, por alimentador.
3. **O DEC e o FEC da empresa por ano** (ponderado pelas UCs: soma das horas ÷ UCs, e não média dos DECs dos alimentadores).
4. **A evolução do pior alimentador** e o evento de julho de 2025.
5. **Agente (P8/P10):** "qual o DEC do alimentador X no último ano?" e "quais os 5 piores alimentadores?".

In [0]:
d = spark.table(tabela_gold)

# 1. Os piores em DEC acumulado de 12 meses (2026-06), com o tamanho do alimentador
pior12 = (d.filter(f"mes_referencia = '{MES_FIM}'")
          .select("alimentador", "municipio", "qtd_ucs_alimentador_mes", "dec_12m_horas", "fec_12m")
          .join(d.filter("mes_referencia BETWEEN '2025-07' AND '2026-06'").groupBy("alimentador")
                 .agg(F.round(F.sum("horas_interrompidas"), 0).alias("horas_12m"),
                      F.round(F.sum("horas_causas_evitaveis"), 0).alias("horas_evitaveis_12m")), "alimentador")
          .withColumn("pct_evitavel", F.round(F.col("horas_evitaveis_12m") / F.col("horas_12m") * 100, 1)))
print("Piores DEC dos últimos 12 meses:")
display(pior12.orderBy(F.desc("dec_12m_horas")).limit(10))
print("Mais horas sem luz (impacto) nos últimos 12 meses:")
display(pior12.orderBy(F.desc("horas_12m")).limit(10))

# Tamanho dos alimentadores (para ler o ranking com cuidado)
display(pior12.agg(F.min("qtd_ucs_alimentador_mes").alias("min_ucs"),
                   F.expr("percentile_approx(qtd_ucs_alimentador_mes, 0.5)").alias("mediana_ucs"),
                   F.max("qtd_ucs_alimentador_mes").alias("max_ucs")))

In [0]:
# 2. Onde a manutenção preventiva dá mais retorno: horas evitáveis nos últimos 12 meses
display(pior12.orderBy(F.desc("horas_evitaveis_12m"))
        .select("alimentador", "municipio", "qtd_ucs_alimentador_mes", "horas_evitaveis_12m", "horas_12m", "pct_evitavel").limit(10))

# Causa principal mais frequente nos meses de cada alimentador (últimos 12 meses)
display(d.filter("mes_referencia BETWEEN '2025-07' AND '2026-06' AND causa_principal IS NOT NULL")
        .groupBy("causa_principal").agg(F.count("*").alias("alimentador_meses")).orderBy(F.desc("alimentador_meses")))

In [0]:
# 3. DEC e FEC da empresa por ano (ponderado: soma das horas / UCs; meses de cada ano somados)
emp = (d.groupBy("mes_referencia").agg(F.sum("horas_interrompidas").alias("h"), F.sum("qtd_interrupcoes").alias("n"),
                                       F.sum("qtd_ucs_alimentador_mes").alias("ucs"))
       .withColumn("dec", F.col("h") / F.col("ucs")).withColumn("fec", F.col("n") / F.col("ucs"))
       .withColumn("ano", F.substring("mes_referencia", 1, 4)))
display(emp.groupBy("ano").agg(F.count("*").alias("meses"), F.round(F.sum("dec"), 2).alias("dec_ano_horas"),
                               F.round(F.sum("fec"), 2).alias("fec_ano")).orderBy("ano"))
# 2022 tem só 6 meses; 2026, só 6

# 4. Julho de 2025 e a evolução do alimentador com o pior DEC de 12 meses
display(d.filter("mes_referencia = '2025-07'").orderBy("ranking_dec_mes")
        .select("ranking_dec_mes", "alimentador", "municipio", "qtd_ucs_alimentador_mes", "qtd_interrupcoes", "dec_horas", "fec", "causa_principal").limit(5))
pior = pior12.orderBy(F.desc("dec_12m_horas")).first()["alimentador"]
print("Alimentador com pior DEC de 12 meses:", pior)
display(d.filter(F.col("alimentador") == pior).select("mes_referencia", "qtd_ucs_alimentador_mes", "qtd_interrupcoes",
        "dec_horas", "dec_12m_horas", "causa_principal", "ranking_dec_mes").orderBy("mes_referencia"))

# 5. Agente: "quais os 5 piores alimentadores em DEC nos últimos 12 meses?"
display(pior12.orderBy(F.desc("dec_12m_horas")).select("alimentador", "municipio", "dec_12m_horas", "qtd_ucs_alimentador_mes").limit(5))